In [2]:
"""Design and develop a distributed application to find the coolest/hottest year from the available
weather data. Use weather data from the Internet and process it using MapReduce. """


import pandas as pd

df = pd.read_csv('mumbai_weather.csv')
df.head()
df['tavg'] = df['tavg'].fillna((df['tmin'] + df['tmax'])/2)
df.isnull().sum()
df['Year'] = df['time'].apply(lambda x: x.split('-')[2])

df.tail()
df['tavg'] = df.groupby('Year')['tavg'].transform(lambda x: x.fillna(x.mean()))
df['tmin'] = df['tmin'].fillna(df['tavg'] - 5)
df['tmax'] = df['tmax'].fillna(df['tavg'] + 5)

df['prcp'] = df['prcp'].fillna(0)
df.isnull().sum()
df.shape
def mapper(df):
    result = []

    for _,row in df.iterrows():
        year = row['Year']
        tavg = row['tavg']
        result.append((year,tavg))

    return result
mapped_output = mapper(df)
mapped_output


ModuleNotFoundError: No module named 'pandas'

In [3]:
def reducer(mapped_output):
    grouped = {}

    for year,tavg in mapped_output:
        if year in grouped:
            grouped[year].append(tavg)
        else:
            grouped[year] = [tavg]

    avg_by_year = {year: sum(tavgs)/len(tavgs) for year,tavgs in grouped.items()}

    return avg_by_year
reducer_ouptput = reducer(mapped_output)
reducer_ouptput
hottest_year = max(reducer_ouptput,key=reducer_ouptput.get)
print("Hottest year:", hottest_year, "with avg temp =", reducer_ouptput[hottest_year])
coldest_year = min(reducer_ouptput,key=reducer_ouptput.get)
print("Coldest year:", coldest_year, "with avg temp =", reducer_ouptput[coldest_year])


NameError: name 'mapped_output' is not defined